# 00_setup — Initialisation de l'environnement

**Projet :** Swiss Hydro & Power Analytics

Ce notebook crée la structure Unity Catalog du projet : les schémas medallion et le Volume de landing.
Il ne charge aucune donnée et peut être relancé sans risque.

## 1. Paramètres

Le catalogue et le nom du projet sont des paramètres (widgets) affichés en haut du notebook.
Le même code peut ainsi être exécuté sur un autre catalogue (dev, test, prod) sans modification.


In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalogue")
dbutils.widgets.text("project", "energy", "Projet")

CATALOG = dbutils.widgets.get("catalog")
PROJECT = dbutils.widgets.get("project")

print(CATALOG, PROJECT)

## 2. Création des schémas medallion

Le projet est organisé en quatre schémas dans le catalogue :

| Schéma | Contenu |
|---|---|
| `energy_raw` | Volume `landing` : fichiers bruts reçus des sources, jamais modifiés |
| `energy_bronze` | Tables Delta brutes avec métadonnées d'ingestion |
| `energy_silver` | Données nettoyées, typées, horodatées en UTC |
| `energy_gold` | KPI métier prêts pour Power BI et l'analyse |

`raw` contient des fichiers ; bronze, silver et gold contiennent des tables Delta.
La commande est idempotente : elle peut être relancée sans erreur.


In [0]:
LAYERS = ["raw", "bronze", "silver", "gold"]

for layer in LAYERS:
    schema = f"{CATALOG}.{PROJECT}_{layer}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print(f"✔ {schema}")

%md
## 3. Volume de landing

Le Volume `landing`, placé dans le schéma `energy_raw`, reçoit les fichiers bruts tels que les sources les envoient (CSV, XML, JSON).
Ces fichiers ne sont jamais modifiés : toute transformation se fait dans les couches bronze, silver et gold.

Chaque source dispose de son propre dossier, ce qui facilitera l'ingestion automatique (Auto Loader).

In [0]:
# Nom complet du Volume au format catalogue.schéma.volume
VOLUME = f"{CATALOG}.{PROJECT}_raw.landing"

# Création du Volume (sans erreur s'il existe déjà)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {VOLUME}")

# Chemin d'accès aux fichiers du Volume, utilisé pour lire et écrire des fichiers
LANDING_PATH = f"/Volumes/{CATALOG}/{PROJECT}_raw/landing"

print(f"✔ {VOLUME}")
print(f"  Chemin : {LANDING_PATH}")

In [0]:
# Un dossier par source et par jeu de données
SOURCES = [
    "entsoe/generation",     # production par filière
    "entsoe/prices",         # prix day-ahead
    "ofev/discharge",        # débits des rivières
    "meteoswiss/weather",    # données météo
    "ofen/reservoirs",       # remplissage des lacs d'accumulation
]

for source in SOURCES:
    # Création du dossier (et des dossiers parents si nécessaire)
    dbutils.fs.mkdirs(f"{LANDING_PATH}/{source}")
    print(f"✔ {LANDING_PATH}/{source}")

## 4. Vérification

Contrôle final : les quatre schémas du projet existent et l'arborescence du Volume est en place.


In [0]:
# Affiche les schémas du catalogue dont le nom commence par le préfixe du projet
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE '{PROJECT}_*'"))

In [0]:
# Affiche les dossiers créés à la racine du Volume landing
display(dbutils.fs.ls(LANDING_PATH))